# 필수 라이브러리

In [4]:
pip install torch transformers pandas scikit-learn kobert-tokenizer

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement kobert-tokenizer (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for kobert-tokenizer


In [5]:
!pip install kobert-tokenizer

ERROR: Could not find a version that satisfies the requirement kobert-tokenizer (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for kobert-tokenizer


In [3]:
# 1. GPU 사용 가능 여부 확인 및 device 설정
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}") # 'cuda'라고 출력되어야 GPU를 사용하는 것입니다.

현재 사용 중인 디바이스: cuda


# 최종 2만개만

In [3]:
import torch
from torch import nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from transformers import get_linear_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification
# 'transformers'에서 필요한 클래스들을 불러옵니다.
from transformers import AutoTokenizer, BertForSequenceClassification

# =========================================
# 클래스 및 함수 정의
# =========================================
# PyTorch Dataset 클래스 정의 (안정성을 높인 버전)
class NSMCDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        # .iloc을 사용하기 위해 데이터프레임의 인덱스를 리셋합니다.
        self.data = dataframe.reset_index(drop=True)
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # .iloc[idx]를 사용해 순서에 상관없이 정확한 행을 가져옵니다.
        row = self.data.iloc[idx]
        sentence = str(row['document'])
        label = int(row['label'])
        
        encoding = self.tokenizer.encode_plus(
            sentence, add_special_tokens=True, max_length=self.max_len,
            return_token_type_ids=False, padding='max_length',
            truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def predict_sentiment(sentence, model, tokenizer, device, max_len=80):
    model.eval()
    encoding = tokenizer.encode_plus(
        sentence, add_special_tokens=True, max_length=max_len,
        return_token_type_ids=False, padding='max_length',
        truncation=True, return_attention_mask=True, return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
    return {
        "sentence": sentence,
        "prediction": "긍정" if prediction == 1 else "부정",
        "positive_score": probs[0][1].item()
    }

# =========================================
# 메인 실행 코드
# =========================================
# 1. 설정 및 디바이스 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}")

MODEL_NAME = "skt/kobert-base-v1"

# 2. 모델 및 토크나이저 로드
print("모델과 토크나이저를 로드합니다...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)  # 중요
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)
print("로드 완료.")

# 3. 데이터셋 준비

print("NSMC 데이터를 로드하고 전처리합니다...")

try:

    train_df = pd.read_csv('nsmc/ratings_train.txt', sep='\t')

    test_df = pd.read_csv('nsmc/ratings_test.txt', sep='\t')

    

    train_df.dropna(inplace=True); train_df.drop_duplicates(subset=['document'], inplace=True)

    test_df.dropna(inplace=True); test_df.drop_duplicates(subset=['document'], inplace=True)



    # ✅ 빠른 테스트를 위해 데이터 축소 (15,000개 / 5,000개)

    train_df = train_df.sample(n=15000, random_state=42)

    test_df = test_df.sample(n=5000, random_state=42)



    print(f"훈련 데이터 {len(train_df)}개, 테스트 데이터 {len(test_df)}개로 축소 완료.")

except FileNotFoundError:

    print("오류: 'ratings_train.txt' 또는 'ratings_test.txt' 파일을 찾을 수 없습니다.")

    exit()

# 4. 학습 설정 (Best-Practice)
MAX_LEN = 64
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 3e-5

train_dataset = NSMCDataset(train_df, tokenizer, MAX_LEN)
test_dataset = NSMCDataset(test_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)


# 전체 훈련 스텝 수 계산
total_steps = len(train_loader) * EPOCHS
# warmup 스텝 수 설정 (보통 전체의 10% 내외, 여기서는 0으로 설정)
warmup_steps = 0 

# 스케줄러 생성
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps=warmup_steps,
                                            num_training_steps=total_steps)

현재 사용 중인 디바이스: cuda
모델과 토크나이저를 로드합니다...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


로드 완료.
NSMC 데이터를 로드하고 전처리합니다...
훈련 데이터 15000개, 테스트 데이터 5000개로 축소 완료.


In [5]:
# 5. 모델 학습
print("\n모델 학습을 시작합니다...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}", leave=True)
    
    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        # ✅ 3. 스케줄러 업데이트
        scheduler.step() 
        
        progress_bar.set_postfix({'loss': loss.item()})
    
    avg_train_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch + 1} 평균 학습 Loss: {avg_train_loss:.4f}")

print("\n학습 완료!")

# 6. 모델 평가
print("\n테스트 데이터로 모델 평가를 시작합니다...")
model.eval()
predictions, true_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).flatten()
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())
accuracy = accuracy_score(true_labels, predictions)
print(f"\n테스트 정확도: {accuracy:.4f}")

# 7. 모델 저장
print("\n학습된 모델과 토크나이저를 저장합니다...")
SAVE_PATH = "./my_kobert_best_model_sample" # 원본 모델과 겹치지 않게 다른 이름으로 저장
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"모델이 '{SAVE_PATH}' 폴더에 성공적으로 저장되었습니다.")

# 8. 저장된 모델로 예측 테스트
print("\n--- 저장된 모델로 새로운 문장 분석 테스트 ---")
my_sentence = "이 영화 정말 재미있고 감동적이었어요. 추천합니다!"
result = predict_sentiment(my_sentence, model, tokenizer, device, max_len=MAX_LEN)
print(f"문장: \"{result['sentence']}\"")
print(f"  >> 예측: {result['prediction']} (긍정 확률: {result['positive_score']:.2%})")


모델 학습을 시작합니다...


Epoch 1/3: 100%|██████████| 469/469 [03:11<00:00,  2.45it/s, loss=0.346]



Epoch 1 평균 학습 Loss: 0.4974


Epoch 2/3: 100%|██████████| 469/469 [03:12<00:00,  2.43it/s, loss=0.235] 



Epoch 2 평균 학습 Loss: 0.3255


Epoch 3/3: 100%|██████████| 469/469 [03:13<00:00,  2.43it/s, loss=0.228] 



Epoch 3 평균 학습 Loss: 0.2217

학습 완료!

테스트 데이터로 모델 평가를 시작합니다...


Evaluating: 100%|██████████| 157/157 [00:20<00:00,  7.56it/s]



테스트 정확도: 0.8408

학습된 모델과 토크나이저를 저장합니다...
모델이 './my_kobert_best_model_sample' 폴더에 성공적으로 저장되었습니다.

--- 저장된 모델로 새로운 문장 분석 테스트 ---
문장: "이 영화 정말 재미있고 감동적이었어요. 추천합니다!"
  >> 예측: 긍정 (긍정 확률: 98.98%)


# 최종(반복 5) 전체 모델 만들기 

In [1]:
import torch
from torch import nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from transformers import get_linear_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification
# 'transformers'에서 필요한 클래스들을 불러옵니다.
from transformers import AutoTokenizer, BertForSequenceClassification

# =========================================
# 클래스 및 함수 정의
# =========================================
# PyTorch Dataset 클래스 정의 (안정성을 높인 버전)
class NSMCDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        # .iloc을 사용하기 위해 데이터프레임의 인덱스를 리셋합니다.
        self.data = dataframe.reset_index(drop=True)
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # .iloc[idx]를 사용해 순서에 상관없이 정확한 행을 가져옵니다.
        row = self.data.iloc[idx]
        sentence = str(row['document'])
        label = int(row['label'])
        
        encoding = self.tokenizer.encode_plus(
            sentence, add_special_tokens=True, max_length=self.max_len,
            return_token_type_ids=False, padding='max_length',
            truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def predict_sentiment(sentence, model, tokenizer, device, max_len=80):
    model.eval()
    encoding = tokenizer.encode_plus(
        sentence, add_special_tokens=True, max_length=max_len,
        return_token_type_ids=False, padding='max_length',
        truncation=True, return_attention_mask=True, return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
    return {
        "sentence": sentence,
        "prediction": "긍정" if prediction == 1 else "부정",
        "positive_score": probs[0][1].item()
    }

# =========================================
# 메인 실행 코드
# =========================================
# 1. 설정 및 디바이스 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}")

MODEL_NAME = "skt/kobert-base-v1"

# 2. 모델 및 토크나이저 로드
print("모델과 토크나이저를 로드합니다...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)  # 중요
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)
print("로드 완료.")

# 3. 데이터셋 준비
print("NSMC 전체 데이터를 로드하고 전처리합니다...")
try:
    train_df = pd.read_csv('nsmc/ratings_train.txt', sep='\t')
    test_df = pd.read_csv('nsmc/ratings_test.txt', sep='\t')
    
    train_df.dropna(inplace=True); train_df.drop_duplicates(subset=['document'], inplace=True)
    test_df.dropna(inplace=True); test_df.drop_duplicates(subset=['document'], inplace=True)

    print(f"훈련 데이터 {len(train_df)}개, 테스트 데이터 {len(test_df)}개 로드 완료.")
except FileNotFoundError:
    print("오류: 'ratings_train.txt' 또는 'ratings_test.txt' 파일을 찾을 수 없습니다.")
    exit()

# 4. 학습 설정 (Best-Practice)
MAX_LEN = 64
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 3e-5

train_dataset = NSMCDataset(train_df, tokenizer, MAX_LEN)
test_dataset = NSMCDataset(test_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)


# 전체 훈련 스텝 수 계산
total_steps = len(train_loader) * EPOCHS
# warmup 스텝 수 설정 (보통 전체의 10% 내외, 여기서는 0으로 설정)
warmup_steps = 0 

# 스케줄러 생성
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps=warmup_steps,
                                            num_training_steps=total_steps)

현재 사용 중인 디바이스: cuda
모델과 토크나이저를 로드합니다...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


로드 완료.
NSMC 전체 데이터를 로드하고 전처리합니다...
훈련 데이터 146182개, 테스트 데이터 49157개 로드 완료.


In [3]:
    # 5. 모델 학습
    print("\n모델 학습을 시작합니다...")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}", leave=True)
        
        for batch in progress_bar:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            progress_bar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = total_loss / len(train_loader)
        print(f"\nEpoch {epoch + 1} 평균 학습 Loss: {avg_train_loss:.4f}")

    print("\n학습 완료!")

    # 6. 모델 평가
    print("\n테스트 데이터로 모델 평가를 시작합니다...")
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).flatten()
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    accuracy = accuracy_score(true_labels, predictions)
    print(f"\n테스트 정확도: {accuracy:.4f}")

    # 7. 모델 저장
    print("\n학습된 모델과 토크나이저를 저장합니다...")
    SAVE_PATH = "./my_kobert_best_model_full" 
    model.save_pretrained(SAVE_PATH)
    tokenizer.save_pretrained(SAVE_PATH)
    print(f"모델이 '{SAVE_PATH}' 폴더에 성공적으로 저장되었습니다.")

    # 8. 저장된 모델로 예측 테스트
    print("\n--- 저장된 모델로 새로운 문장 분석 테스트 ---")
    my_sentence = "이 영화 정말 재미있고 감동적이었어요. 추천합니다!"
    result = predict_sentiment(my_sentence, model, tokenizer, device, max_len=MAX_LEN)
    print(f"문장: \"{result['sentence']}\"") 
    print(f"  >> 예측: {result['prediction']} (긍정 확률: {result['positive_score']:.2%})")


모델 학습을 시작합니다...


Epoch 1/5: 100%|██████████| 4569/4569 [29:17<00:00,  2.60it/s, loss=0.115] 



Epoch 1 평균 학습 Loss: 0.3653


Epoch 2/5: 100%|██████████| 4569/4569 [29:51<00:00,  2.55it/s, loss=0.0966]



Epoch 2 평균 학습 Loss: 0.2656


...


Epoch 4 평균 학습 Loss: 0.1470


Epoch 5/5: 100%|██████████| 4569/4569 [44:21<00:00,  1.72it/s, loss=0.0131] 



Epoch 5 평균 학습 Loss: 0.1051

학습 완료!

테스트 데이터로 모델 평가를 시작합니다...


Evaluating: 100%|██████████| 1537/1537 [04:35<00:00,  5.58it/s]



테스트 정확도: 0.8854

학습된 모델과 토크나이저를 저장합니다...
모델이 './my_kobert_best_model_full' 폴더에 성공적으로 저장되었습니다.

--- 저장된 모델로 새로운 문장 분석 테스트 ---
문장: "이 영화 정말 재미있고 감동적이었어요. 추천합니다!"
  >> 예측: 긍정 (긍정 확률: 99.27%)


# 성능 재평가(ppt 담으려고)

In [4]:
pip install sentencepiece


  Using cached sentencepiece-0.2.1-cp312-cp312-win_amd64.whl.metadata (10 kB)
Using cached sentencepiece-0.2.1-cp312-cp312-win_amd64.whl (1.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
# -*- coding: utf-8 -*-
import os
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from tqdm import tqdm

# --------------------------
# 설정 (모델만 불러와 NSMC 재평가)
# --------------------------
MODEL_DIR   = "./my_kobert_best_model_full"   # 저장했던 모델 폴더
NSMC_TRAIN  = "nsmc/ratings_train.txt"
NSMC_TEST   = "nsmc/ratings_test.txt"

MAX_LEN     = 64
BATCH_SIZE  = 64       # GPU면 64~128, CPU면 32 권장
NUM_WORKERS = 0        # Windows/Jupyter 멈춤 방지
PIN_MEMORY  = False
VAL_RATIO   = 0.1
SEED        = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------
# 체크
# --------------------------
if not os.path.isdir(MODEL_DIR):
    raise FileNotFoundError(f"모델 폴더가 없습니다: {MODEL_DIR}")
if not os.path.isfile(NSMC_TRAIN) or not os.path.isfile(NSMC_TEST):
    raise FileNotFoundError("NSMC 파일을 찾을 수 없습니다. 경로를 확인하세요.")

# --------------------------
# 모델/토크나이저 로드
# --------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()
print(f"[INFO] Model loaded from {MODEL_DIR} | device={DEVICE}")




[INFO] Model loaded from ./my_kobert_best_model_full | device=cuda


In [4]:
# --------------------------
# 데이터 로드
# --------------------------
def load_nsmc(path):
    df = pd.read_csv(path, sep="\t").dropna()
    df = df.drop_duplicates(subset=["document"]).reset_index(drop=True)
    return df

train_df = load_nsmc(NSMC_TRAIN)
test_df  = load_nsmc(NSMC_TEST)

# train/valid 분리 (라벨 비율 유지)
train_df, valid_df = train_test_split(
    train_df, test_size=VAL_RATIO, random_state=SEED, stratify=train_df["label"]
)

print(f"[INFO] train={len(train_df):,}, valid={len(valid_df):,}, test={len(test_df):,}")

# --------------------------
# Dataset / collate_fn (배치 토크나이즈)
# --------------------------
class NSMCDataset(Dataset):
    def __init__(self, df):
        self.texts = df["document"].astype(str).tolist()
        self.labels = df["label"].astype(int).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {"text": self.texts[i], "label": self.labels[i]}

def collate_fn(batch):
    texts = [b["text"] for b in batch]
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    enc = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}


[INFO] train=131,563, valid=14,619, test=49,157


In [6]:
# --------------------------
# DataLoader
# --------------------------
train_loader = DataLoader(NSMCDataset(train_df), batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)
valid_loader = DataLoader(NSMCDataset(valid_df), batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)
test_loader  = DataLoader(NSMCDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)

# --------------------------
# 평가 (진행률바 + 러닝 ACC 표시)
# --------------------------
@torch.inference_mode()
def eval_with_progress(dataloader, model, device, desc):
    model.eval()
    preds, trues = [], []
    correct, total = 0, 0
    bar = tqdm(dataloader, desc=desc, leave=True)
    for batch in bar:
        ids  = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        y    = batch["labels"].to(device, non_blocking=True)

        logits = model(ids, attention_mask=mask).logits
        pred = torch.argmax(logits, dim=1)

        preds.extend(pred.cpu().numpy())
        trues.extend(y.cpu().numpy())

        correct += (pred == y).sum().item()
        total   += y.size(0)
        bar.set_postfix(acc=f"{(correct/total)*100:.2f}%")

    acc = accuracy_score(trues, preds)
    p, r, f1, _ = precision_recall_fscore_support(trues, preds, average="macro", zero_division=0)
    cm = confusion_matrix(trues, preds)
    return acc, p, r, f1, cm

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

print("\n[INFO] 재평가 시작...")
tr_acc, _, _, _, _            = eval_with_progress(train_loader, model, DEVICE, "Train   eval")
va_acc, _, _, _, _            = eval_with_progress(valid_loader, model, DEVICE, "Valid   eval")
te_acc, te_p, te_r, te_f1, cm = eval_with_progress(test_loader,  model, DEVICE, "Test    eval")

# --------------------------
# 출력 (핵심: NSMC ACC)
# --------------------------
print("\n====================== NSMC 재평가 결과 ======================")
print(f"Train ACC : {tr_acc:.4f}")
print(f"Valid ACC : {va_acc:.4f}")
print(f"Test  ACC : {te_acc:.4f}")         # <- 발표에 쓸 핵심 숫자
print(f"(참고) Macro-F1: {te_f1:.4f} | Precision: {te_p:.4f} | Recall: {te_r:.4f}")
print("Confusion Matrix (Test):\n", cm)

# --------------------------
# (선택) 한 문장 예측 데모
# --------------------------
@torch.inference_mode()
def predict_one(text):
    enc = tokenizer(text, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)
    probs = torch.softmax(model(ids, attention_mask=mask).logits, dim=1).squeeze(0)
    pred  = torch.argmax(probs).item()
    return ("긍정" if pred==1 else "부정", float(probs[1]))

demo = "이 영화 정말 재미있고 감동적이었어요. 추천합니다!"
pred, pos = predict_one(demo)
print(f'\n[데모] "{demo}" → {pred} (긍정 확률: {pos:.2%})')


[INFO] 재평가 시작...


Test    eval: 100%|██████████| 769/769 [03:08<00:00,  4.08it/s, acc=88.54%]


====================== NSMC 재평가 결과 ======================
Train ACC : 0.9832
Valid ACC : 0.9846
Test  ACC : 0.8854
(참고) Macro-F1: 0.8854 | Precision: 0.8854 | Recall: 0.8854
Confusion Matrix (Test):
 [[21541  2905]
 [ 2728 21983]]

[데모] "이 영화 정말 재미있고 감동적이었어요. 추천합니다!" → 긍정 (긍정 확률: 99.27%)
